In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")                 # 无显示器环境下的绘图后端
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
s_true = 1.1
n_vocab = 20000                       # 词表容量（潜在词型总数）
ranks = np.arange(1, n_vocab + 1)
p = ranks ** (-s_true)
p /= p.sum()                          # 归一化成词频分布
N = 100000                            # 合成语料总 token 数
counts = rng.multinomial(N, p)

keep = counts > 0
r_obs = ranks[keep].astype(float)
f_obs = counts[keep].astype(float)

slope, intercept = np.polyfit(np.log(r_obs), np.log(f_obs), 1)
print(f"实际出现的词型数 = {int(keep.sum())}")
print(f"全部观测的双对数回归斜率 = {slope:.3f}")

hi = f_obs >= 10                      # 只用高频头部拟合，避开采样截断
slope_hi, intercept_hi = np.polyfit(np.log(r_obs[hi]), np.log(f_obs[hi]), 1)
print(f"仅用频次 >= 10 的头部词型拟合斜率 = {slope_hi:.3f}")
print(f"仅出现 1 次的词型数 = {int((counts == 1).sum())}")

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.loglog(r_obs, f_obs, ".", ms=2, alpha=0.4, label="observed")
ax.loglog(r_obs, np.exp(intercept) * r_obs ** slope, "-", lw=1.5,
          label=f"fit slope={slope:.2f}")
ax.set_xlabel("rank r"); ax.set_ylabel("frequency f(r)")
ax.legend(frameon=False)
fig.savefig("zipf.png", dpi=150)

实际出现的词型数 = 10538
全部观测的双对数回归斜率 = -0.765
仅用频次 >= 10 的头部词型拟合斜率 = -1.013
仅出现 1 次的词型数 = 5281


In [2]:
import hashlib
import numpy as np

rng = np.random.default_rng(42)

def shingles(text, k=5):
    """k-shingling：把文本转成长度为 k 的字符片段集合"""
    return {text[i:i + k] for i in range(len(text) - k + 1)}

def jaccard(A, B):
    return len(A & B) / len(A | B)

base = "统计学家使用随机抽样估计总体参数并对估计量的方差进行评估以给出置信区间"
dup = base.replace("评估", "评价")          # 近重复（改写两词）
unrel = "机器学习模型通过最小化训练集上的经验风险来学习参数"

def h(item, seed, p=(1 << 61) - 1):
    """第 seed 个哈希函数：sha1 映射到素数域，模拟一次随机置换"""
    digest = hashlib.sha1(str(seed).encode() + item.encode()).digest()
    return int.from_bytes(digest[:8], "big") % p

def minhash_signature(S, k=64):
    return [min(h(x, s) for x in S) for s in range(k)]

print("== 单对文档的 Jaccard 估计 ==")
for a, b, label in [(base, dup, "近重复对"), (base, unrel, "无关对")]:
    A, B = shingles(a), shingles(b)
    J = jaccard(A, B)
    sigA = np.array(minhash_signature(A))
    sigB = np.array(minhash_signature(B))
    Jhat = (sigA == sigB).mean()
    print(f"{label}: 精确 J = {J:.3f}, 64 维签名估计 = {Jhat:.3f}")

J_true, k, R = 0.8, 64, 200000
agree = rng.random((R, k)) < J_true     # 每个哈希位置一致与否 ~ Bernoulli(J)
Jhats = agree.mean(axis=1)
print("== 估计量性质的 Monte Carlo 验证 ==")
print(f"真实 J = {J_true}, {R} 次重复: 估计均值 = {Jhats.mean():.4f}")
print(f"方差: 模拟 = {Jhats.var():.6f}, 理论 J(1-J)/k = {J_true*(1-J_true)/k:.6f}")
print(f"标准误: 模拟 = {Jhats.std():.4f}, 理论 = {np.sqrt(J_true*(1-J_true)/k):.4f}")

b, r = 16, 4    # k = br = 64, 曲线中点 s* = (1/b)^(1/r) = 0.5
def candidate_prob(s, trials=100000):
    agree = rng.random((trials, b * r)) < s
    bands = agree.reshape(trials, b, r)
    return (bands.all(axis=2).any(axis=1)).mean()

print(f"== banding 候选概率 (b={b}, r={r}) ==")
for s in [0.2, 0.4, 0.5, 0.6, 0.8, 0.9]:
    sim = candidate_prob(s)
    theo = 1 - (1 - s**r)**b
    print(f"s={s:.1f}: 模拟候选概率 = {sim:.4f}, 理论 = {theo:.4f}")

== 单对文档的 Jaccard 估计 ==
近重复对: 精确 J = 0.722, 64 维签名估计 = 0.719
无关对: 精确 J = 0.000, 64 维签名估计 = 0.000
== 估计量性质的 Monte Carlo 验证 ==
真实 J = 0.8, 200000 次重复: 估计均值 = 0.8000
方差: 模拟 = 0.002484, 理论 J(1-J)/k = 0.002500
标准误: 模拟 = 0.0498, 理论 = 0.0500
== banding 候选概率 (b=16, r=4) ==
s=0.2: 模拟候选概率 = 0.0248, 理论 = 0.0253
s=0.4: 模拟候选概率 = 0.3409, 理论 = 0.3396
s=0.5: 模拟候选概率 = 0.6431, 理论 = 0.6439
s=0.6: 模拟候选概率 = 0.8909, 理论 = 0.8915
s=0.8: 模拟候选概率 = 0.9998, 理论 = 0.9998
s=0.9: 模拟候选概率 = 1.0000, 理论 = 1.0000


In [3]:
import numpy as np

normal_sentences = [
    "统计学通过数据推断总体规律",
    "回归分析用于刻画变量之间的关系",
    "抽样调查需要考虑覆盖误差与无回答误差",
    "置信区间给出了估计的不确定性范围",
    "假设检验控制第一类错误的发生概率",
    "极大似然估计在正则条件下具有相合性",
    "贝叶斯方法将先验分布与数据结合",
    "时间序列分析关注观测之间的自相关结构",
]
train = "".join(normal_sentences * 4)

chars = sorted(set(train))
idx = {c: i for i, c in enumerate(chars)}
V = len(chars)

# 字符级 bigram，加一平滑（Laplace smoothing）
big = np.ones((V + 1, V + 1))          # 第 V 行/列留给训练集外的字符
for a, b in zip(train, train[1:]):
    big[idx[a], idx[b]] += 1
big /= big.sum(axis=1, keepdims=True)

def ppl(text):
    logp = []
    for a, b in zip(text, text[1:]):
        ia = idx.get(a, V)
        ib = idx.get(b, V)
        logp.append(np.log2(big[ia, ib]))
    return 2 ** (-np.mean(logp))

docs = {
    "同域正常文档": "假设检验的第一类错误概率由显著性水平控制",
    "同域换序文档": "的率检检误错一第由制平水显性著差控",
    "乱码文档":     "j迷#3宫z^x@乱~码$%片&段*组(合)",
}
for name, text in docs.items():
    print(f"{name}: PPL = {ppl(text):.2f}")

THRESH = 80.0
for name, text in docs.items():
    verdict = "保留" if ppl(text) < THRESH else "剔除"
    print(f"阈值 {THRESH}: {name} -> {verdict}")

同域正常文档: PPL = 49.62
同域换序文档: PPL = 106.41
乱码文档: PPL = 102.37
阈值 80.0: 同域正常文档 -> 保留
阈值 80.0: 同域换序文档 -> 剔除
阈值 80.0: 乱码文档 -> 剔除


In [ ]:
# from datatrove.executor.local import LocalPipelineExecutor
# from datatrove.pipeline.readers import JsonlReader
# from datatrove.pipeline.filters import (
#     GopherQualityFilter,          # Gopher 论文整理的启发式规则集
#     LanguageFilter,               # fastText 语种判别，此处保留中英文
# )
# from datatrove.pipeline.writers import JsonlWriter

# pipeline = [
#     JsonlReader("data/raw/", text_key="text"),              # 读取原始 jsonl / warc / parquet
#     LanguageFilter(language_threshold=0.65, languages=["zh", "en"]),
#     GopherQualityFilter(min_doc_words=80, max_doc_words=20000),
#     JsonlWriter("data/cleaned/"),                            # 输出清洗后语料
# ]

# executor = LocalPipelineExecutor(pipeline=pipeline, workers=16, tasks=16)
# executor.run()  # 每个 worker 处理一个 shard，天然并行

In [2]:
import re
# 依据 Baker & Smith, Gopher (DeepMind, 2021) 公开的启发式规则整理
def gopher_like_filter(text: str) -> dict:
    words = text.split()
    n = len(words)
    if not (50 <= n <= 100000):                       # 文档长度上下限
        return {"keep": False, "reason": "length"}
    if len(text) / max(n, 1) > 100:                   # 平均词长过长，疑似乱码或表格
        return {"keep": False, "reason": "avg_word_len"}
    if text.count("...") > 10 or text.count("…") > 10:  # 省略号堆叠，疑似截断/占位
        return {"keep": False, "reason": "ellipsis"}
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    dup_line = 1 - len(set(lines)) / max(len(lines), 1)
    if dup_line > 0.30:                               # 重复行占比过高，疑似模板页
        return {"keep": False, "reason": "dup_lines"}
    bullets = sum(l.startswith(("*", "-", "•")) for l in lines)
    if bullets / max(len(lines), 1) > 0.90:           # 通篇列表，多为目录或索引
        return {"keep": False, "reason": "bullet_list"}
    return {"keep": True, "reason": "pass"}

In [3]:
from collections import Counter, defaultdict

def train_bpe(corpus, num_merges):
    """corpus: Dict[str, int]，键为单词、值为频次；执行固定 num_merges 次合并。
    每次合并恰好向词表新增一个符号，因此合并步数与词表大小一一对应。"""
    vocab = Counter()                       # step 1: 词频统计
    for word, freq in corpus.items():
        vocab[" ".join(list(word)) + " </w>"] += freq

    merges = []                             # 记录每一步的合并规则，用于推理阶段复现
    for _ in range(num_merges):
        pairs = Counter()
        for token_seq, freq in vocab.items():
            symbols = token_seq.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq   # step 2: 统计相邻符号对频次
        if not pairs:
            break
        best = max(pairs, key=pairs.get)                      # step 3: 取频次最高的一对
        merges.append(best)
        vocab = merge_pair(best, vocab)                       # step 4: 全局替换并回到 step 2
    return merges, get_token_vocab(vocab)

def merge_pair(pair, vocab):
    """把词表中所有 token 序列里出现的 pair 合并为一个符号"""
    new_vocab, pat, rep = Counter(), " ".join(pair), "".join(pair)
    for token_seq, freq in vocab.items():
        new_vocab[token_seq.replace(pat, rep)] += freq
    return new_vocab

def get_token_vocab(vocab):
    tokens = set()
    for token_seq in vocab:
        tokens.update(token_seq.split())
    return tokens

# 推理：对新单词按训练阶段学到的合并顺序贪心执行
def encode(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:                        # 按学习顺序而非频次二次排序
        i, out = 0, []
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == (a, b):
                out.append(a + b); i += 2
            else:
                out.append(symbols[i]); i += 1
        symbols = out
    return symbols

In [4]:
corpus = {
    "low": 5, "lower": 2, "lowest": 6, "newest": 6, "widest": 3,
    "统计": 8, "统计学": 5, "统计学家": 2, "数学": 4, "数学家": 3,
}
print(f"初始符号集大小 = {len(set(''.join(corpus)) | {'</w>'})}")
merges, vocab = train_bpe(corpus, num_merges=12)
for i, (a, b) in enumerate(merges, 1):
    print(f"  合并 {i:2d}: '{a}' + '{b}'")
print(encode("统计学", merges))
print(encode("newest", merges))
print(encode("lowers", merges))

初始符号集大小 = 16
  合并  1: 'e' + 's'
  合并  2: 'es' + 't'
  合并  3: 'est' + '</w>'
  合并  4: '统' + '计'
  合并  5: 'l' + 'o'
  合并  6: 'lo' + 'w'
  合并  7: '学' + '</w>'
  合并  8: '统计' + '</w>'
  合并  9: 'low' + 'est</w>'
  合并 10: 'n' + 'e'
  合并 11: 'ne' + 'w'
  合并 12: 'new' + 'est</w>'
['统计', '学</w>']
['newest</w>']
['low', 'e', 'r', 's', '</w>']


In [1]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
ids = enc.encode("在因果推断中，倾向得分匹配用于平衡协变量分布。")
print(len(ids), round(len("在因果推断中，倾向得分匹配用于平衡协变量分布。") / len(ids), 2))

27 0.85
